In [1]:
#from langchain_community.document_loaders import PyPDFLoader

from langchain_community.document_loaders import UnstructuredPDFLoader

/Users/anikparui/Desktop/Project/RAG/rag_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loader = UnstructuredPDFLoader("M_Tech_Project.pdf")

In [3]:
data = loader.load()

No languages specified, defaulting to English.


In [4]:
data

[Document(metadata={'source': 'M_Tech_Project.pdf'}, page_content='INDIAN INSTITUTE OF TECHNOLOGY, KHARAGPUR\n\nStock Price Prediction using Reproducing Kernel Hilbert Space\n\nA seminar project submitted for the fulfillment of the requirements for the degree of Master in Technology (M.Tech.)\n\nAnik Parui M.Tech. 1st year (2nd Sem) Roll No. :- 24MA60R22 Department of Mathematics\n\nSupervisor Dr. Chandal Nahak Department of Matematics\n\nDr. Pabitra Mitra Department of Computer Science\n\nApril 2025\n\nContents\n\n1 Introduction\n\n2 Function Space\n\n3 Feature space in indefinite dimension\n\n4 Evaluation Functional\n\n5 Dual Space\n\n6 Riesz Representation Theorem\n\n7 Reproducing Kernel\n\n8 Reproducing Kernel Hilbert Space\n\n9 Advantages of Reproducing kernel Hilbert Space\n\n10 Disadvantages of Reproducing Kernel Hilbert Space\n\n11 code\n\n12 Result\n\n13 future plan\n\n14 Reference\n\n1\n\n2\n\n2\n\n3\n\n3\n\n4\n\n4\n\n4\n\n5\n\n6\n\n6\n\n7\n\n10\n\n10\n\n11\n\n1\n\nIntroducti

In [5]:
type(data)

list

In [6]:
print("The length of the data = ", len(data))

The length of the data =  1


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
#split the data
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 100)
docs = text_splitter.split_documents(data)

In [9]:
print("The number of documents = ", len(docs))

The number of documents =  45


In [10]:
#load ing huggingface embedding class
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings()

vector = embeddings.embed_query("hello world!")

/var/folders/49/dlfryms541x6nvjcz5yz0hh80000gn/T/ipykernel_28563/3613637858.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/var/folders/49/dlfryms541x6nvjcz5yz0hh80000gn/T/ipykernel_28563/3613637858.py:3: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 6361.45it/s]


In [11]:
print(len(vector))

768


In [12]:
#creating a vector database to store all the values of vector embeddings
# the vector database is named as vectorstore

from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(docs, embedding = embeddings)

In [13]:
# we want to use this vector database as retriver
# the task of retriver is to use those documents that matches the user's query

retriever = vectorstore.as_retriever(search_type = "similarity", search_kwargs = {"k" : 5})
retrieved_docs = retriever.invoke("stock price")

In [14]:
# printing the length of the retrieved document

print("Length of the retrived document = ", len(retrieved_docs))

Length of the retrived document =  5


In [15]:
# print the content of any document

print(retrieved_docs[2].page_content)

The prediction of asset price is a challenging task. Firstly, financial time series are very difficult to predict because of their nonstationarity and noisiness. The challenge presented by the ”efficient market hypothesis”, according to which there is not room for non-obvious price prediction in efficient markets. At the same time rapid expansion of fully automated, algorithmic trading systems increases the demand for precise models of real-world mar- ket prices. For predicting financial market


In [18]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

# Use Groq API key to load the groq llm
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key = api_key, model_name = "llama-3.3-70b-versatile")

prompt_template = """
You are a factual assistant.

Answer the question ONLY using the context below.
- Do NOT use prior knowledge
- Do NOT refuse unless context is empty
- If answer exists in context, give it directly

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template
)

# Chain
llm_chain = prompt | llm | StrOutputParser()

In [19]:
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
    
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | llm_chain
)

In [20]:
#Input question
question = "what are the advantages of RKHS ?"

#answer
response = rag_chain.invoke(question)
print(response)

The advantages of RKHS are:

1. Efficient Learning: Using kernels, RKHS methods can operate implicitly in high dimensions without actually computing the coordinates.
2. Compatibility with Regularization Techniques: Regularized learning works very naturally within RKHS, improving robustness to noisy financial data.
3. Handling Nonlinear Relationships: RKHS allows mapping data into a higher-dimensional space where nonlinear patterns become linear and more manageable.
4. Flexibility through Choice of Kernel: Different kernels can be chosen depending on the nature of the stock market data.
